# YOLO11s Football Ball Detector Training

- Model: `yolo11s.pt`
- Epochs: `150`
- Persistent Google Drive project folder, so training can resume after a Colab disconnect
- Checkpoints saved every epoch with `save_period=1`
- Auto-resume from `last.pt` if it exists
- Same football-specific augmentation implementation:
  - Linear motion blur with random angles from `0°` to `360°`
  - Random scale/crop/pad from `0.5x` to `1.5x`
  - HSV shifts
  - Random brightness/contrast
  - YOLO Mosaic on
  - YOLO MixUp off
  - Multi-scale training on

## Before running

1. In Colab, go to **Runtime → Change runtime type → GPU**
2. Run every cell from top to bottom.
3. Put your dataset zip in Drive at:
   `/content/drive/MyDrive/football_yolo11s_colab/dataset_zip/ball-dataset.zip`

If that zip is not found, the notebook will ask you to upload one and will copy it into Drive so future restarts are easier.


In [ ]:
# =========================
# 0. INSTALL PACKAGES
# =========================

!pip -q install -U ultralytics opencv-python pyyaml tqdm


In [ ]:
# =========================
# 1. MOUNT DRIVE + IMPORTS + GPU CHECK
# =========================

from google.colab import drive
drive.mount("/content/drive")

import os
import glob
import shutil
import zipfile
import random
from pathlib import Path

import cv2
import yaml
import numpy as np
from tqdm import tqdm
import torch
from ultralytics import YOLO
from IPython.display import FileLink, display

print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("GPU count:", torch.cuda.device_count())

if torch.cuda.is_available():
    print("GPU 0:", torch.cuda.get_device_name(0))
else:
    raise RuntimeError(
        "No GPU found. In Colab, go to Runtime → Change runtime type → GPU, then restart."
    )


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Torch: 2.11.0+cu128
CUDA available: True
GPU count: 1
GPU 0: Tesla T4


In [ ]:
# =========================
# 2. CONFIG
# =========================

# Training settings
MODEL_NAME = "yolo11s.pt"
RUN_NAME = "football_detector_yolo11s_150_aug"
EPOCHS = 150
IMAGE_SIZE = 640
BATCH_SIZE = 16
PATIENCE = 30
RANDOM_SEED = 42

# Validation split if your dataset only has train/
VAL_FRACTION = 0.15

# Persistent Drive project folder
DRIVE_PROJECT_DIR = Path("/content/drive/MyDrive/football_yolo11s_colab")
DATASET_ZIP_PATH = Path("/content/drive/MyDrive/NFL Detection 1500.yolov11.zip")

# Optional: set this to a Drive folder containing data.yaml if you do not want to use a zip.
# Example: DATASET_FOLDER_PATH = "/content/drive/MyDrive/my_roboflow_dataset"
DATASET_FOLDER_PATH = ""

# Ephemeral local working folders rebuilt each runtime.
# This prevents duplicate augmentations on rerun/restart.
WORK_DIR = Path("/content/football_yolo11s_work")
RAW_DIR = WORK_DIR / "raw_dataset"
AUG_DIR = WORK_DIR / "aug_dataset"

# Persistent training outputs/checkpoints.
RUNS_DIR = DRIVE_PROJECT_DIR / "runs"
RUN_DIR = RUNS_DIR / RUN_NAME
WEIGHTS_DIR = RUN_DIR / "weights"
LAST_PT = WEIGHTS_DIR / "last.pt"
BEST_PT = WEIGHTS_DIR / "best.pt"

IMG_EXTS = [".jpg", ".jpeg", ".png", ".bmp", ".webp"]

# -------------------------
# Offline custom augmentations
# -------------------------

# Keep horizontal flip from previous implementation.
USE_HORIZONTAL_FLIP = True
HORIZONTAL_FLIP_PROB = 0.5

# 1) High-speed linear motion blur
USE_MOTION_BLUR = True
MOTION_BLUR_PROB = 0.5
MOTION_BLUR_MIN_LENGTH = 5
MOTION_BLUR_MAX_LENGTH = 60
MOTION_BLUR_MIN_ANGLE = 0.0
MOTION_BLUR_MAX_ANGLE = 360.0

# 2) Resolution simulation + scaling/crop
USE_RANDOM_SCALE_CROP = True
RANDOM_SCALE_PROB = 1.0
RANDOM_SCALE_MIN = 0.50
RANDOM_SCALE_MAX = 1.50

# 4) HSV + lighting robustness
USE_HSV = True
HUE_SHIFT_DEG = 10            # random +/- 10 degrees on OpenCV HSV hue scale
SATURATION_MIN = 0.75         # -25%
SATURATION_MAX = 1.25         # +25%
VALUE_MIN = 0.85              # -15%
VALUE_MAX = 1.15              # +15%

USE_BRIGHTNESS_CONTRAST = True
CONTRAST_MIN = 0.85           # -15%
CONTRAST_MAX = 1.15           # +15%
BRIGHTNESS_DELTA = 0.15       # +/- 15% of 255

print("Persistent project dir:", DRIVE_PROJECT_DIR)
print("Dataset zip path:", DATASET_ZIP_PATH)
print("Run dir:", RUN_DIR)
print("Resume checkpoint:", LAST_PT)


Persistent project dir: /content/drive/MyDrive/football_yolo11s_colab
Dataset zip path: /content/drive/MyDrive/NFL Detection 1500.yolov11.zip
Run dir: /content/drive/MyDrive/football_yolo11s_colab/runs/football_detector_yolo11s_150_aug
Resume checkpoint: /content/drive/MyDrive/football_yolo11s_colab/runs/football_detector_yolo11s_150_aug/weights/last.pt


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# =========================
# 3. PREPARE DATASET FROM DRIVE/UPLOAD
# =========================

from google.colab import files

DRIVE_PROJECT_DIR.mkdir(parents=True, exist_ok=True)
DATASET_ZIP_PATH.parent.mkdir(parents=True, exist_ok=True)
RUNS_DIR.mkdir(parents=True, exist_ok=True)

# Fresh local work dirs every runtime/rerun.
# Outputs and checkpoints remain safe in Drive.
if WORK_DIR.exists():
    shutil.rmtree(WORK_DIR)
RAW_DIR.mkdir(parents=True, exist_ok=True)
AUG_DIR.mkdir(parents=True, exist_ok=True)

def find_data_yaml(root: Path):
    yamls = sorted(root.glob("**/data.yaml"))
    return yamls[0] if yamls else None

if DATASET_FOLDER_PATH:
    dataset_folder = Path(DATASET_FOLDER_PATH)
    if not dataset_folder.exists():
        raise FileNotFoundError(f"DATASET_FOLDER_PATH does not exist: {dataset_folder}")
    if find_data_yaml(dataset_folder) is None:
        raise FileNotFoundError(f"No data.yaml found inside DATASET_FOLDER_PATH: {dataset_folder}")
    print("Copying dataset folder from Drive:", dataset_folder)
    shutil.copytree(dataset_folder, RAW_DIR, dirs_exist_ok=True)

else:
    if not DATASET_ZIP_PATH.exists():
        print("Dataset zip not found in Drive.")
        print("Upload your Roboflow YOLO dataset zip now. It will be saved to Drive for restarts.")
        uploaded = files.upload()

        zip_names = [name for name in uploaded.keys() if name.lower().endswith(".zip")]
        if not zip_names:
            raise FileNotFoundError("No .zip file uploaded.")

        uploaded_zip = Path("/content") / zip_names[0]
        shutil.copy(uploaded_zip, DATASET_ZIP_PATH)
        print("Saved uploaded zip to:", DATASET_ZIP_PATH)

    print("Extracting dataset zip:", DATASET_ZIP_PATH)
    with zipfile.ZipFile(DATASET_ZIP_PATH, "r") as zip_ref:
        zip_ref.extractall(RAW_DIR)

DATA_YAML = find_data_yaml(RAW_DIR)
if DATA_YAML is None:
    raise FileNotFoundError("Could not find data.yaml after preparing dataset.")

DATASET_ROOT = DATA_YAML.parent
print("Found data.yaml:", DATA_YAML)
print("Dataset root:", DATASET_ROOT)

# Copy raw dataset into augmentable local dataset.
shutil.copytree(DATASET_ROOT, AUG_DIR, dirs_exist_ok=True)

print("Copied dataset to local augmentable dir:", AUG_DIR)


Extracting dataset zip: /content/drive/MyDrive/NFL Detection 1500.yolov11.zip
Found data.yaml: /content/football_yolo11s_work/raw_dataset/data.yaml
Dataset root: /content/football_yolo11s_work/raw_dataset
Copied dataset to local augmentable dir: /content/football_yolo11s_work/aug_dataset


In [ ]:
# =========================
# 4. NORMALIZE FOLDER NAMES + CREATE VALIDATION IF MISSING
# =========================

def get_images(folder: Path):
    paths = []
    if not folder.exists():
        return []
    for ext in IMG_EXTS:
        paths.extend(folder.glob(f"*{ext}"))
        paths.extend(folder.glob(f"*{ext.upper()}"))
    return sorted(list(set(paths)))

# Some datasets use val/ instead of valid/.
if (AUG_DIR / "val").exists() and not (AUG_DIR / "valid").exists():
    shutil.move(str(AUG_DIR / "val"), str(AUG_DIR / "valid"))

train_img_dir = AUG_DIR / "train" / "images"
train_label_dir = AUG_DIR / "train" / "labels"
valid_img_dir = AUG_DIR / "valid" / "images"
valid_label_dir = AUG_DIR / "valid" / "labels"

if not train_img_dir.exists():
    raise FileNotFoundError(f"Could not find train images folder: {train_img_dir}")

if not train_label_dir.exists():
    raise FileNotFoundError(f"Could not find train labels folder: {train_label_dir}")

# If no validation set exists, split 15% from train.
if not valid_img_dir.exists():
    print("No valid/images folder found. Creating validation split from train...")

    valid_img_dir.mkdir(parents=True, exist_ok=True)
    valid_label_dir.mkdir(parents=True, exist_ok=True)

    train_images = get_images(train_img_dir)
    random.seed(RANDOM_SEED)
    random.shuffle(train_images)

    n_valid = max(1, int(len(train_images) * VAL_FRACTION))
    valid_images = train_images[:n_valid]

    for img_path in valid_images:
        label_path = train_label_dir / f"{img_path.stem}.txt"

        shutil.move(str(img_path), str(valid_img_dir / img_path.name))

        if label_path.exists():
            shutil.move(str(label_path), str(valid_label_dir / label_path.name))
        else:
            # Keep empty label file for negative / no-ball images.
            (valid_label_dir / f"{img_path.stem}.txt").write_text("")

else:
    valid_label_dir.mkdir(parents=True, exist_ok=True)

# Make sure labels exist for all images. Empty label files are valid YOLO negatives.
for img_dir, label_dir in [(train_img_dir, train_label_dir), (valid_img_dir, valid_label_dir)]:
    label_dir.mkdir(parents=True, exist_ok=True)
    for img_path in get_images(img_dir):
        label_path = label_dir / f"{img_path.stem}.txt"
        if not label_path.exists():
            label_path.write_text("")

print("Train images:", len(get_images(train_img_dir)))
print("Valid images:", len(get_images(valid_img_dir)))


Train images: 1069
Valid images: 214


In [ ]:
# =========================
# 5. YOLO LABEL HELPERS
# =========================

def read_yolo_label(label_path: Path):
    boxes = []

    if not label_path.exists():
        return boxes

    with open(label_path, "r") as f:
        for line in f:
            parts = line.strip().split()

            if len(parts) != 5:
                continue

            cls, x, y, w, h = parts
            boxes.append([int(float(cls)), float(x), float(y), float(w), float(h)])

    return boxes


def write_yolo_label(label_path: Path, boxes):
    label_path.parent.mkdir(parents=True, exist_ok=True)

    with open(label_path, "w") as f:
        for cls, x, y, w, h in boxes:
            x = min(max(float(x), 0.0), 1.0)
            y = min(max(float(y), 0.0), 1.0)
            w = min(max(float(w), 0.0), 1.0)
            h = min(max(float(h), 0.0), 1.0)

            if w <= 0 or h <= 0:
                continue

            f.write(f"{int(cls)} {x:.6f} {y:.6f} {w:.6f} {h:.6f}\n")


def yolo_to_xyxy(boxes, img_w, img_h):
    xyxy = []
    for cls, x, y, w, h in boxes:
        x1 = (x - w / 2) * img_w
        y1 = (y - h / 2) * img_h
        x2 = (x + w / 2) * img_w
        y2 = (y + h / 2) * img_h
        xyxy.append([cls, x1, y1, x2, y2])
    return xyxy


def xyxy_to_yolo(xyxy, img_w, img_h, min_box_px=2):
    boxes = []
    for cls, x1, y1, x2, y2 in xyxy:
        x1 = min(max(x1, 0), img_w)
        y1 = min(max(y1, 0), img_h)
        x2 = min(max(x2, 0), img_w)
        y2 = min(max(y2, 0), img_h)

        bw = x2 - x1
        bh = y2 - y1

        if bw < min_box_px or bh < min_box_px:
            continue

        x = (x1 + x2) / 2 / img_w
        y = (y1 + y2) / 2 / img_h
        w = bw / img_w
        h = bh / img_h

        boxes.append([cls, x, y, w, h])

    return boxes


In [ ]:
# =========================
# 6. CUSTOM AUGMENTATION FUNCTIONS
# =========================

def horizontal_flip(image, boxes):
    flipped = cv2.flip(image, 1)

    new_boxes = []
    for cls, x, y, w, h in boxes:
        new_boxes.append([cls, 1.0 - x, y, w, h])

    return flipped, new_boxes


def linear_motion_blur(image, length=50, angle=0.0):
    # Make odd kernel length for clean center.
    length = int(max(3, length))
    if length % 2 == 0:
        length += 1

    kernel = np.zeros((length, length), dtype=np.float32)
    kernel[length // 2, :] = 1.0
    kernel /= length

    center = (length / 2 - 0.5, length / 2 - 0.5)
    rot_mat = cv2.getRotationMatrix2D(center, angle, 1.0)
    kernel = cv2.warpAffine(kernel, rot_mat, (length, length))
    kernel_sum = kernel.sum()
    if kernel_sum > 0:
        kernel /= kernel_sum

    return cv2.filter2D(image, -1, kernel)


def random_scale_crop_pad(image, boxes, scale_min=0.50, scale_max=1.50):
    h, w = image.shape[:2]
    scale = random.uniform(scale_min, scale_max)

    new_w = max(1, int(round(w * scale)))
    new_h = max(1, int(round(h * scale)))

    resized = cv2.resize(image, (new_w, new_h), interpolation=cv2.INTER_LINEAR)

    xyxy = yolo_to_xyxy(boxes, w, h)
    scaled_xyxy = []
    for cls, x1, y1, x2, y2 in xyxy:
        scaled_xyxy.append([cls, x1 * scale, y1 * scale, x2 * scale, y2 * scale])

    # If zoomed in, random crop back to original size.
    if scale >= 1.0:
        max_x = max(0, new_w - w)
        max_y = max(0, new_h - h)
        crop_x = random.randint(0, max_x) if max_x > 0 else 0
        crop_y = random.randint(0, max_y) if max_y > 0 else 0

        out = resized[crop_y:crop_y + h, crop_x:crop_x + w]

        shifted_xyxy = []
        for cls, x1, y1, x2, y2 in scaled_xyxy:
            shifted_xyxy.append([cls, x1 - crop_x, y1 - crop_y, x2 - crop_x, y2 - crop_y])

        new_boxes = xyxy_to_yolo(shifted_xyxy, w, h)

    # If zoomed out, pad back to original size with random placement.
    else:
        out = np.zeros_like(image)
        # Use median turf/stadium-ish color from the image border instead of pure black.
        border_pixels = np.concatenate([
            image[0, :, :],
            image[-1, :, :],
            image[:, 0, :],
            image[:, -1, :]
        ], axis=0)
        pad_color = np.median(border_pixels, axis=0).astype(np.uint8)
        out[:, :] = pad_color

        max_x = max(0, w - new_w)
        max_y = max(0, h - new_h)
        pad_x = random.randint(0, max_x) if max_x > 0 else 0
        pad_y = random.randint(0, max_y) if max_y > 0 else 0

        out[pad_y:pad_y + new_h, pad_x:pad_x + new_w] = resized

        shifted_xyxy = []
        for cls, x1, y1, x2, y2 in scaled_xyxy:
            shifted_xyxy.append([cls, x1 + pad_x, y1 + pad_y, x2 + pad_x, y2 + pad_y])

        new_boxes = xyxy_to_yolo(shifted_xyxy, w, h)

    return out, new_boxes


def adjust_hsv(image):
    hsv = cv2.cvtColor(image, cv2.COLOR_BGR2HSV).astype(np.float32)

    hue_shift = random.uniform(-HUE_SHIFT_DEG, HUE_SHIFT_DEG)
    sat_factor = random.uniform(SATURATION_MIN, SATURATION_MAX)
    val_factor = random.uniform(VALUE_MIN, VALUE_MAX)

    hsv[:, :, 0] = (hsv[:, :, 0] + hue_shift) % 180
    hsv[:, :, 1] *= sat_factor
    hsv[:, :, 2] *= val_factor

    hsv[:, :, 1] = np.clip(hsv[:, :, 1], 0, 255)
    hsv[:, :, 2] = np.clip(hsv[:, :, 2], 0, 255)

    hsv = hsv.astype(np.uint8)
    return cv2.cvtColor(hsv, cv2.COLOR_HSV2BGR)


def random_brightness_contrast(image):
    alpha = random.uniform(CONTRAST_MIN, CONTRAST_MAX)
    beta = random.uniform(-BRIGHTNESS_DELTA * 255, BRIGHTNESS_DELTA * 255)

    out = image.astype(np.float32) * alpha + beta
    return np.clip(out, 0, 255).astype(np.uint8)


def augment_one_image(image, boxes):
    # Random scale/crop/pad first so boxes are geometrically updated before blur/color.
    if USE_RANDOM_SCALE_CROP and random.random() < RANDOM_SCALE_PROB:
        image, boxes = random_scale_crop_pad(
            image,
            boxes,
            scale_min=RANDOM_SCALE_MIN,
            scale_max=RANDOM_SCALE_MAX,
        )

    if USE_HORIZONTAL_FLIP and random.random() < HORIZONTAL_FLIP_PROB:
        image, boxes = horizontal_flip(image, boxes)

    if USE_HSV:
        image = adjust_hsv(image)

    if USE_BRIGHTNESS_CONTRAST:
        image = random_brightness_contrast(image)

    if USE_MOTION_BLUR and random.random() < MOTION_BLUR_PROB:
        length = random.randint(MOTION_BLUR_MIN_LENGTH, MOTION_BLUR_MAX_LENGTH)
        angle = random.uniform(MOTION_BLUR_MIN_ANGLE, MOTION_BLUR_MAX_ANGLE)
        image = linear_motion_blur(image, length=length, angle=angle)

    return image, boxes


In [ ]:
# =========================
# 7. APPLY OFFLINE AUGMENTATIONS TO TRAIN ONLY
# =========================

train_images_before = get_images(train_img_dir)

# Only augment non-augmented originals. Since AUG_DIR is rebuilt every runtime,
# this prevents duplicate _aug files even if you rerun this cell.
train_images_to_augment = [p for p in train_images_before if "_aug" not in p.stem]

print("Training originals to augment:", len(train_images_to_augment))

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

augmented_count = 0
for img_path in tqdm(train_images_to_augment):
    image = cv2.imread(str(img_path))

    if image is None:
        print("Skipping unreadable image:", img_path)
        continue

    label_path = train_label_dir / f"{img_path.stem}.txt"
    boxes = read_yolo_label(label_path)

    aug_image, aug_boxes = augment_one_image(image, boxes)

    aug_img_path = train_img_dir / f"{img_path.stem}_aug{img_path.suffix}"
    aug_label_path = train_label_dir / f"{img_path.stem}_aug.txt"

    cv2.imwrite(str(aug_img_path), aug_image)
    write_yolo_label(aug_label_path, aug_boxes)
    augmented_count += 1

print("Augmented images created:", augmented_count)
print("After augmentation:")
print("Train images:", len(get_images(train_img_dir)))
print("Valid images:", len(get_images(valid_img_dir)))


Training originals to augment: 1069


100%|██████████| 1069/1069 [03:53<00:00,  4.58it/s]

Augmented images created: 1069
After augmentation:
Train images: 2138
Valid images: 214


In [ ]:
# =========================
# 8. FIX data.yaml ABSOLUTE PATHS
# =========================

aug_yaml_path = AUG_DIR / "data.yaml"

with open(aug_yaml_path, "r") as f:
    data = yaml.safe_load(f)

data["train"] = str(AUG_DIR / "train" / "images")
data["val"] = str(AUG_DIR / "valid" / "images")

if (AUG_DIR / "test" / "images").exists():
    data["test"] = str(AUG_DIR / "test" / "images")

with open(aug_yaml_path, "w") as f:
    yaml.safe_dump(data, f, sort_keys=False)

print("Final data.yaml:")
print(open(aug_yaml_path).read())


Final data.yaml:
train: /content/football_yolo11s_work/aug_dataset/train/images
val: /content/football_yolo11s_work/aug_dataset/valid/images
test: /content/football_yolo11s_work/aug_dataset/test/images
nc: 2
names:
- '0'
- '1'
roboflow:
  workspace: scotts-workspace-vjp1v
  project: scotts-workspace-vjp1v
  version: dataset
  license: Private
  url: https://app.roboflow.com/scotts-workspace-vjp1v/scotts-workspace-vjp1v/dataset



In [ ]:
# =========================
# 9. DELETE OLD YOLO CACHE FILES + SANITY CHECK
# =========================

cache_files = list(AUG_DIR.glob("**/*.cache"))

for cache_file in cache_files:
    cache_file.unlink()
    print("Deleted cache:", cache_file)

train_img_count = len(get_images(AUG_DIR / "train" / "images"))
train_label_count = len(list((AUG_DIR / "train" / "labels").glob("*.txt")))
valid_img_count = len(get_images(AUG_DIR / "valid" / "images"))
valid_label_count = len(list((AUG_DIR / "valid" / "labels").glob("*.txt")))

print("========== FINAL DATASET COUNTS ==========")
print("Train images:", train_img_count)
print("Train labels:", train_label_count)
print("Valid images:", valid_img_count)
print("Valid labels:", valid_label_count)
print("==========================================")

if train_img_count == 0 or valid_img_count == 0:
    raise RuntimeError("Train or validation set is empty.")

if train_img_count != train_label_count:
    print("Warning: train image/label count mismatch. Empty labels should have been created.")

if valid_img_count != valid_label_count:
    print("Warning: valid image/label count mismatch. Empty labels should have been created.")


========== FINAL DATASET COUNTS ==========
Train images: 2138
Train labels: 2138
Valid images: 214
Valid labels: 214


In [ ]:
# =========================
# 10. TRAIN YOLO11s FOR 150 EPOCHS, AUTO-RESUME SAFE
# =========================

# Augmentation list:
# 1. High-speed motion blur: implemented offline above with random angle 0-360 deg.
# 2. Resolution simulation/scaling/crop: implemented offline above with 0.5x-1.5x scale/crop/pad.
# 3. Mosaic: YOLO built-in mosaic=1.0 below.
# 4. MixUp: YOLO built-in mixup=0.0 below so the tiny ball does not disappear.
# 5. HSV + lighting: implemented offline above and YOLO hsv_* below.
# 6. Random brightness/contrast: implemented offline above.
# 7. Multi-scale: YOLO multi_scale=0.50 below, giving 0.5x-1.5x batch image-size variation.

train_kwargs = dict(
    data=str(aug_yaml_path),
    epochs=EPOCHS,
    imgsz=IMAGE_SIZE,
    batch=BATCH_SIZE,
    device=0,
    patience=PATIENCE,

    project=str(RUNS_DIR),
    name=RUN_NAME,
    exist_ok=True,

    save=True,
    save_period=1,

    # Built-in YOLO augmentation settings optimized for small football detection
    hsv_h=0.015,
    hsv_s=0.25,
    hsv_v=0.15,
    translate=0.05,
    scale=0.50,
    multi_scale=0.50,
    fliplr=0.5,

    mosaic=1.00,
    close_mosaic=10,
    mixup=0.00,
    cutmix=0.00,
    erasing=0.00,

    plots=True,
    verbose=True,
)

if LAST_PT.exists():
    print("Found resume checkpoint:", LAST_PT)
    print("Resuming training from last.pt")
    model = YOLO(str(LAST_PT))

    # Ultralytics restores optimizer, scheduler, and epoch when resume=True.
    # Passing epochs=EPOCHS keeps the target run length at 150 in current versions.
    results = model.train(resume=True, epochs=EPOCHS)

else:
    print("No checkpoint found. Starting new training from:", MODEL_NAME)
    model = YOLO(MODEL_NAME)
    results = model.train(**train_kwargs)

print("Training complete.")
print("Best weights:", BEST_PT)
print("Last checkpoint:", LAST_PT)


No checkpoint found. Starting new training from: yolo11s.pt
Ultralytics 8.4.60 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/football_yolo11s_work/aug_dataset/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=150, erasing=0.0, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.25, hsv_v=0.15, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11s.pt, momentum=0.937, mosaic=1.0, multi_scale=0.5, name=football_detector_yo

In [ ]:
# =========================
# 11. VALIDATE BEST MODEL
# =========================

if BEST_PT.exists():
    val_model = YOLO(str(BEST_PT))
else:
    print("best.pt not found yet, validating current model object instead.")
    val_model = model

metrics = val_model.val(data=str(aug_yaml_path), imgsz=IMAGE_SIZE, device=0)
print(metrics)
print("Best weights path:", BEST_PT)


Ultralytics 8.4.60 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
YOLO11s summary (fused): 101 layers, 9,413,574 parameters, 0 gradients, 21.3 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 3627.1±299.3 MB/s, size: 465.0 KB)
val: Scanning /content/football_yolo11s_work/aug_dataset/valid/labels.cache... 214 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 214/214 81.6Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 14/14 2.4it/s 5.9s
                   all        214       4639      0.939        0.9      0.954      0.742
                     0        213       4266      0.985       0.92      0.976      0.763
                     1        175        373      0.893      0.879      0.932      0.721
Speed: 2.0ms preprocess, 6.4ms inference, 0.0ms loss, 3.0ms postprocess per image
Results saved to /content/runs/detect/val
ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_i

In [ ]:
# =========================
# 12. EXPORT EASY DOWNLOAD COPY + ZIP TO DRIVE
# =========================

EXPORT_DIR = DRIVE_PROJECT_DIR / "exports"
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

if not BEST_PT.exists():
    raise FileNotFoundError(f"No best.pt found at {BEST_PT}. Finish training first.")

easy_best = EXPORT_DIR / "best_yolo11s_football.pt"
shutil.copy(BEST_PT, easy_best)

zip_path = EXPORT_DIR / "MUST_DOWNLOAD_yolo11s_weights.zip"
if zip_path.exists():
    zip_path.unlink()

with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as z:
    z.write(BEST_PT, arcname="best.pt")
    if LAST_PT.exists():
        z.write(LAST_PT, arcname="last.pt")

print("Copied best weights to:", easy_best)
print("Created zip:", zip_path)

# Also copy to /content for direct Colab file browser/download convenience.
local_best = Path("/content/best_yolo11s_football.pt")
local_zip = Path("/content/MUST_DOWNLOAD_yolo11s_weights.zip")
shutil.copy(easy_best, local_best)
shutil.copy(zip_path, local_zip)

display(FileLink(str(local_best)))
display(FileLink(str(local_zip)))


Copied best weights to: /content/drive/MyDrive/football_yolo11s_colab/exports/best_yolo11s_football.pt
Created zip: /content/drive/MyDrive/football_yolo11s_colab/exports/MUST_DOWNLOAD_yolo11s_weights.zip


/content/best_yolo11s_football.pt

/content/MUST_DOWNLOAD_yolo11s_weights.zip

## How to restart safely

If Colab disconnects:

1. Reopen this notebook.
2. Run cells from the top.
3. The dataset zip and `runs/football_detector_yolo11s_150_aug/weights/last.pt` live in Google Drive.
4. The training cell automatically resumes from `last.pt`.

Because the local working dataset is rebuilt from the Drive zip every runtime, rerunning the augmentation cell will not keep stacking duplicate `_aug_aug_aug` files.
